In [ ]:
import sqlite3
import pandas as pd
from collections import Counter, defaultdict
from itertools import combinations
import random
import math
from pathlib import Path

# ============================================================
# 1. LOAD DATA
# ============================================================

sqlite_path = Path(__file__).resolve().parent / "data" / "ocel2-p2p.sqlite"
conn = sqlite3.connect(sqlite_path)

event_object = pd.read_sql("""
SELECT ocel_event_id, ocel_object_id
FROM event_object
""", conn)

conn.close()

# ============================================================
# 2. BUILD EVENT STRUCTURE
# ============================================================
# Map each event to its set of associated objects: O(e)
event_objects = event_object.groupby("ocel_event_id")["ocel_object_id"].apply(set)

event_df = pd.DataFrame({
    "event_id": event_objects.index,
    "objects": event_objects.values
}).reset_index(drop=True)

print(f"Total events loaded: {len(event_df)}")

# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================
random.seed(42)

train_df = event_df.sample(frac=0.7, random_state=42)
test_df = event_df.drop(train_df.index)

# ============================================================
# 4. COMPUTE GLOBAL SUPPORT & CONDITIONAL PROBABILITIES
# ============================================================
support_singleton = Counter()
support_pair = Counter()

# Count occurrences of singletons and pairs in the training set
for objs in train_df["objects"]:
    for o in objs:
        support_singleton[o] += 1
    for o1, o2 in combinations(objs, 2):
        support_pair[(o1, o2)] += 1
        support_pair[(o2, o1)] += 1

# Calculate \hat{P}(o_2 | o_1) = supp({o1, o2}) / supp({o1})
# We store this as a nested dictionary: conditional_prob[o1][o2] = prob
conditional_prob = defaultdict(dict)

for (o1, o2), pair_supp in support_pair.items():
    sing_supp = support_singleton[o1]
    if sing_supp > 0:
        conditional_prob[o1][o2] = pair_supp / sing_supp

# ============================================================
# 5. CORRUPTION MODEL (Simulate Partially Observed Logs)
# ============================================================
def corrupt(objs, drop_fraction=0.3):
    objs_list = list(objs)
    if len(objs_list) <= 1:
        return set(objs_list)
    k = max(1, int(len(objs_list) * drop_fraction))
    return set(objs_list) - set(random.sample(objs_list, k))

test_df["corrupted"] = test_df["objects"].apply(corrupt)

def predict_ranked(observed_objs):
    """
    Returns candidates ranked by score.
    """

    candidates = set()

    for o in observed_objs:
        for o_prime in conditional_prob[o]:
            if o_prime not in observed_objs:
                candidates.add(o_prime)

    scores = {}

    for o_prime in candidates:

        total_log_prob = 0.0

        for o in observed_objs:
            prob = conditional_prob[o].get(o_prime, 1e-9)
            total_log_prob += math.log(prob)

        scores[o_prime] = total_log_prob

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [obj for obj, score in ranked]

def evaluate_hits_at_k(df, k=10):

    total_missing = 0
    hits = 0

    for _, row in df.iterrows():

        true_set = row["objects"]
        observed = row["corrupted"]

        missing = true_set - observed

        if not missing:
            continue

        ranked_preds = predict_ranked(observed)

        top_k = set(ranked_preds[:k])

        for m in missing:

            total_missing += 1

            if m in top_k:
                hits += 1

    return hits / total_missing if total_missing else 0
# ============================================================
# 6. SCORING AND RECONSTRUCTION (PREDICTION)
# ============================================================
def predict(observed_objs, top_k=3):
    """
    Reconstruct missing objects using Top-K ranking.

    1. Generate candidate pool
    2. Score candidates using summed log probabilities
    3. Return K highest-scoring candidates
    """

    # Candidate generation
    candidates = set()

    for o in observed_objs:
        for o_prime in conditional_prob[o]:
            if o_prime not in observed_objs:
                candidates.add(o_prime)

    if not candidates:
        return set()

    # Candidate scoring
    scores = {}

    for o_prime in candidates:

        total_log_prob = 0.0

        for o in observed_objs:

            prob = conditional_prob[o].get(
                o_prime,
                1e-9
            )

            total_log_prob += math.log(prob)

        scores[o_prime] = total_log_prob

    # Rank candidates
    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Keep only Top-K
    reconstructed = {
        obj
        for obj, _
        in ranked[:top_k]
    }

    return reconstructed

# ============================================================
# 7. EVALUATION
# ============================================================
def evaluate(df, top_k=3, num_samples=3):
    total_missing = 0
    correct = 0
    total_predicted = 0
    samples_printed = 0

    for _, row in df.iterrows():
        true_set = row["objects"]
        observed = row["corrupted"]

        missing = true_set - observed
        if not missing:
            continue

        preds = predict(observed, top_k=top_k)

        total_missing += len(missing)
        total_predicted += len(preds)

        for m in missing:
            if m in preds:
                correct += 1

        # Print sample event details during evaluation
        if samples_printed < num_samples:
            print("--- SAMPLE EVENT ---")
            print(f"True: {true_set}")
            print(f"Observed: {observed}")
            print(f"Predicted: {list(preds)}")
            samples_printed += 1

    recall = correct / total_missing if total_missing else 0
    precision = correct / total_predicted if total_predicted else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) else 0

    return recall, precision, f1

def evaluate_mrr(df):

    reciprocal_ranks = []

    for _, row in df.iterrows():

        true_set = row["objects"]
        observed = row["corrupted"]

        missing = true_set - observed

        if not missing:
            continue

        ranked_preds = predict_ranked(observed)

        for m in missing:

            if m in ranked_preds:

                rank = ranked_preds.index(m) + 1

                reciprocal_ranks.append(
                    1.0 / rank
                )

            else:

                reciprocal_ranks.append(0.0)

    return (
        sum(reciprocal_ranks)
        / len(reciprocal_ranks)
        if reciprocal_ranks else 0
    )
    
for k in [1, 3, 5, 10]:
    score = evaluate_hits_at_k(test_df, k)
    print(f"Hits@{k}: {score:.4f}")

mrr_score = evaluate_mrr(test_df)
print(f"MRR: {mrr_score:.4f}")

def show_examples(df, n=5, top_k=10):

    shown = 0

    for _, row in df.iterrows():

        true_set = row["objects"]
        observed = row["corrupted"]

        missing = true_set - observed

        if not missing:
            continue

        ranked = predict_ranked(observed)
        preds = set(ranked[:top_k])

        print("\n==============================")
        print(f"EVENT {shown+1}")
        print("==============================")

        print("Observed:")
        print(observed)

        print("\nTrue:")
        print(true_set)

        print("\nMissing (ground truth):")
        print(missing)

        print(f"\nTop-{top_k} predictions:")
        print(preds)

        print("\nFull ranking (top 15):")
        print(ranked[:15])

        print("\nHits:")
        print(len(missing.intersection(preds)), "/", len(missing))

        shown += 1

        if shown >= n:
            break
        
show_examples(test_df, n=5, top_k=10)

Total events loaded: 14671
Hits@1: 0.2399
Hits@3: 0.4978
Hits@5: 0.6195
Hits@10: 0.7023
MRR: 0.3909

EVENT 1
Observed:
{'purchase_order:8'}

True:
{'quotation:3', 'purchase_order:8'}

Missing (ground truth):
{'quotation:3'}

Top-10 predictions:
{'goods receipt:5', 'goods receipt:4'}

Full ranking (top 15):
['goods receipt:5', 'goods receipt:4']

Hits:
0 / 1

EVENT 2
Observed:
{'purchase_order:537'}

True:
{'goods receipt:645', 'purchase_order:537'}

Missing (ground truth):
{'goods receipt:645'}

Top-10 predictions:
{'goods receipt:645', 'invoice receipt:651', 'purchase_order:536', 'goods receipt:647', 'goods receipt:644', 'purchase_order:538', 'payment:317', 'quotation:306', 'goods receipt:646'}

Full ranking (top 15):
['goods receipt:646', 'goods receipt:647', 'quotation:306', 'goods receipt:645', 'invoice receipt:651', 'purchase_order:536', 'goods receipt:644', 'purchase_order:538', 'payment:317']

Hits:
1 / 1

EVENT 3
Observed:
{'invoice receipt:640'}

True:
{'invoice receipt:640', 